# Trainen voor de Mystery Device

In [10]:
import numpy as np
from tensorflow.keras.datasets import mnist

from mysterydevice_model import NeuralNetworkTraining, NeuralNetworkInference, sparse_cross_entropy
from read_image import load_image, classify_image, hidden_layer_sizes, output_nodes

## Definiëren van het model en functies

Voordat we het eindprogramma kunnen schrijven, moeten we natuurlijk eerst een model trainen. We hebben al verschillende modellen getraind in de P opdrachten, maar nu moeten we een model kiezen dat goed presteert en binnen de constraints van de Mystery Device past.

Zie `experimenten.ipynb` voor een overzicht van de gemaakte P opdrachten en de resultaten daarvan. Hier wordt ook de keuze voor het eindmodel gemotiveerd.

### Data laden

In [2]:
def load_mnist(npz_path='mnist.npz'):
    (x_train, y_train), (x_test, y_test) = mnist.load_data(path=npz_path)
    return (x_train, y_train), (x_test, y_test)

### Preprocessing

In [3]:
def normalize_data(x):
    return x / 255.0


def flatten_data(x):
    return x.reshape(len(x), -1)

### Model

In [4]:
input_nodes = 28 * 28
output_nodes = 10
dropout_rates = [0.2]
total_models = 12

hidden_layer_sizes = [[110 + (i * 3)] for i in range(total_models)]
subset_size = 0.7

## Trainen van het model

### Inladen en preprocessen van de data

In [5]:
(x_train, y_train), (x_test, y_test) = load_mnist()
x_train = normalize_data(flatten_data(x_train))
x_test = normalize_data(flatten_data(x_test))

### Trainen

In [6]:
from pathlib import Path

npz_files = list(Path(".").glob("weights*.npz"))
total_currently_saved_npz = len(npz_files)

if total_currently_saved_npz != total_models:
    for f in npz_files:
        f.unlink()

    for i in range(total_models):
        print(f"{'=' * 15} Model {i} {'=' * 15}")

        current_hidden_layer_sizes = hidden_layer_sizes[i]

        # Create a unique subset of the dataset
        sample_size = int(len(x_train) * subset_size)

        indices = np.random.choice(len(x_train), size=sample_size, replace=True)
        x_randomized = x_train[indices]
        y_randomized = y_train[indices]

        model = NeuralNetworkTraining(input_nodes, current_hidden_layer_sizes, output_nodes, dropout_rates)
        model.train(x_randomized, y_randomized, learning_rate=0.003, epochs=200, batch_size=16, val_split=0.1, patience=20, verbose=True)
        model.save_weights(f'weights_{i}.npz')

        if i < total_models - 1:
            print(f"\n\n")

=============== Model 0 ===============
Epoch   1 | Acc: 0.8829 | Loss: 0.3925 | Val Loss: 0.2214
Epoch   2 | Acc: 0.9405 | Loss: 0.2020 | Val Loss: 0.1533
Epoch   3 | Acc: 0.9555 | Loss: 0.1523 | Val Loss: 0.1208
Epoch   4 | Acc: 0.9632 | Loss: 0.1257 | Val Loss: 0.1091
Epoch   5 | Acc: 0.9678 | Loss: 0.1075 | Val Loss: 0.0938
Epoch   6 | Acc: 0.9726 | Loss: 0.0938 | Val Loss: 0.0896
Epoch   7 | Acc: 0.9756 | Loss: 0.0846 | Val Loss: 0.0819
Epoch   8 | Acc: 0.9784 | Loss: 0.0728 | Val Loss: 0.0781
Epoch   9 | Acc: 0.9802 | Loss: 0.0662 | Val Loss: 0.0730
Epoch  10 | Acc: 0.9819 | Loss: 0.0613 | Val Loss: 0.0737
Epoch  11 | Acc: 0.9829 | Loss: 0.0586 | Val Loss: 0.0679
Epoch  12 | Acc: 0.9831 | Loss: 0.0554 | Val Loss: 0.0645
Epoch  13 | Acc: 0.9846 | Loss: 0.0509 | Val Loss: 0.0621
Epoch  14 | Acc: 0.9856 | Loss: 0.0474 | Val Loss: 0.0639
Epoch  15 | Acc: 0.9875 | Loss: 0.0430 | Val Loss: 0.0623
Epoch  16 | Acc: 0.9883 | Loss: 0.0404 | Val Loss: 0.0590
Epoch  17 | Acc: 0.9886 | Loss: 

### Test de Models samen

In [7]:
all_probs = []

for i in range(total_models):
    current_hidden_layer_sizes = hidden_layer_sizes[i]

    m = NeuralNetworkInference(f'weights_{i}.npz')
    probs = m.forward(x_test)
    all_probs.append(probs.copy())
    del m

avg_probs = np.mean(all_probs, axis=0)
predicted = np.argmax(avg_probs, axis=1)
acc = np.mean(predicted == y_test)
loss = sparse_cross_entropy(y_test, avg_probs)

print("All Models:")
print(f"Test Accuracy {acc:.4f} | Test Loss: {loss:.4f}")

All Models:
Test Accuracy 0.9819 | Test Loss: 0.0586


### Peak RAM tijdens inference meten

Tijdens inference zijn er meerdere sources die deel uitmaken van het uiteindelijke totaal peak RAM tijdens inference.
Hieronder volgen de verschillende arrays die samen de peak RAM maken.

- Persistente arrays: altijd in RAM
  - Model weights
  - Model biases
  - Min/max per laag
- Tijdelijke arrays tijdens een forward pass
  - Input laag
  - Pre-activation
  - Current
  - Gedequantiseerde gewichten per kolom per laag
  - Output pre-activation
  - Prediction

In [8]:
max_model_idx = np.argmax([h[0] for h in hidden_layer_sizes])
worst_case_hidden_nodes = hidden_layer_sizes[max_model_idx][0]

_m = NeuralNetworkInference(f'weights_{max_model_idx}.npz')

persistent_weights = sum(w.nbytes for w in _m.weights)   # uint8
persistent_biases = sum(b.nbytes for b in _m.biases)     # float32
persistent_minmax = len(_m.w_min) * 2 * 4                # float32 min+max per laag

# _matmul dequantiseert kolom voor kolom; peak = één kolom van de grootste laag (laag 1: 784 floats)
peak_dequantized_col = input_nodes * 4

temporary = (
    1 * input_nodes * 4 +
    1 * worst_case_hidden_nodes * 4 +                    # preac laag 1
    1 * worst_case_hidden_nodes * 4 +                    # current laag 1
    peak_dequantized_col +                               # dequantized gewichtenkolom (peak: laag 1)
    1 * output_nodes * 4 +
    1 * output_nodes * 4
)

persistent = persistent_weights + persistent_biases + persistent_minmax

print(f"Worst-case Model Index:    {max_model_idx} ({worst_case_hidden_nodes} hidden nodes)")
print(f"Gewichten (uint8):         {persistent_weights / 1024:.2f} KB")
print(f"Biases (float32):          {persistent_biases / 1024:.2f} KB")
print(f"Min/max (float32):         {persistent_minmax / 1024:.2f} KB")
print(f"Tijdelijke arrays:         {temporary / 1024:.2f} KB")
print(f"Totaal peak:               {(persistent + temporary) / 1024:.2f} KB")

del _m

Worst-case Model Index:    11 (143 hidden nodes)
Gewichten (uint8):         110.88 KB
Biases (float32):          0.60 KB
Min/max (float32):         0.02 KB
Tijdelijke arrays:         7.32 KB
Totaal peak:               118.81 KB


### Peak RAM tijdens inference meten met tracemalloc

In [92]:
import tracemalloc

tracemalloc.start()

pre = tracemalloc.take_snapshot()
_m = NeuralNetworkInference(f'weights_{max_model_idx}.npz')
snapshot_loaded = tracemalloc.take_snapshot()

test_image = load_image("0.png")

classify_image(test_image)

snapshot_predicted = tracemalloc.take_snapshot()

tracemalloc.stop()
del _m

def total_kb(snapshot):
    return sum(stat.size for stat in snapshot.statistics('lineno')) / 1024

print(f"Begin runtime:   {total_kb(pre):.2f} KB")
print(f"Na laden model:  {total_kb(snapshot_loaded):.2f} KB")
print(f"Na predict:      {total_kb(snapshot_predicted):.2f} KB")

Begin runtime:   0.89 KB
Na laden model:  122.48 KB
Na predict:      213.17 KB


> De bovenstaande meting met tracemalloc geeft een indicatie van het RAM-gebruik tijdens verschillende fasen van de runtime.
> - Na het laden van het model zien we een aanzienlijke toename in RAM-gebruik, wat overeenkomt met de opslag van de modelgewichten, biases en min/max waarden.
>   - Het model zou nooit permanent in RAM staan zoals ik hier meet, maar dit geeft een indicatie van wat er in RAM ongeveer gebeurt aan het begin van `classify_image()`.
> - Na het uitvoeren van `classify_image()` zien we een verdere toename in RAM-gebruik, wat overeenkomt met de tijdelijke arrays die worden aangemaakt tijdens de forward pass.
> - In `classify_image()` worden de modellen met gc verwijderd na gebruik, zodat ze niet lui in RAM blijven staan.

### Totale opslag meten

In [93]:
from pathlib import Path

npz_files = sorted(Path(".").glob("weights*.npz"))

total = 0
for f in npz_files:
    kb = f.stat().st_size / 1024
    total += kb
    print(f"{f.name:<25} {kb:>8.2f} KB")

print("-" * 37)
print(f"{'Totaal':<25} {total:>8.2f} KB")

if npz_files:
    gemiddelde = total / len(npz_files)
    print(f"{'Gemiddelde':<25} {gemiddelde:>8.2f} KB")

print(f"{'Percentage van 1024 KB':<25} {total / 1024 * 100:>7.1f}%")

weights_0.npz                69.33 KB
weights_1.npz                70.73 KB
weights_10.npz               87.85 KB
weights_11.npz               87.52 KB
weights_2.npz                73.15 KB
weights_3.npz                73.87 KB
weights_4.npz                75.95 KB
weights_5.npz                77.78 KB
weights_6.npz                80.08 KB
weights_7.npz                81.09 KB
weights_8.npz                84.55 KB
weights_9.npz                85.58 KB
-------------------------------------
Totaal                      947.48 KB
Gemiddelde                   78.96 KB
Percentage van 1024 KB       92.5%


### Gemiddelde herkenningstijd van 1 image

In [94]:
import time

test_image = load_image("0.png")

runs = 100
start = time.perf_counter()
for _ in range(runs):
    classify_image(test_image)
end = time.perf_counter()

avg_ms = (end - start) / runs * 1000
print(f"Gemiddelde tijd per classificatie: {avg_ms:.2f} ms (over {runs} runs)")

Gemiddelde tijd per classificatie: 9.62 ms (over 100 runs)
